# RigTech · runner_colab_continuacao — DINOv3 + Transformer

**Filosofia do ciclo**: o modelo é **instrumento fixo de auditoria** — mesmo backbone (DINOv3-ViTL16 satélite, congelado), mesma head (2 camadas Transformer sobre patch tokens), mesmos hiperparâmetros. O que muda entre ciclos é o **dataset** (correções humanas em `Datasets/DaninhasTreinoClientes/`). Cada rodada produz um `.pt` versionado (`ciclo_N`) que serve pra medir o quanto as correções melhoraram o rótulo.

**Task**: classificação binária por bloco de 16px (`cultivo=0` / `daninha=1` / `ignore=255`). As 3 classes de daninha do YOLO antigo (folha_larga, folha_estreita, mamona) são colapsadas em `daninha`.

**Fluxo**:
1. DINOv3 congelado extrai features por tile de 512px (grid 32×32 = 1024 tokens de 1024 dim cada) → checkpoint por site.
2. Split treino/val: faixa espacial dentro de cada fazenda (20% coluna direita = val, margem 1 tile). TODAS as fazendas contribuem pra treino E val.
3. Head Transformer (2 layers, 8 heads, GELU, norm_first) prediz `cultivo`/`daninha` por bloco.
4. Loss `ce_dice` (CE ponderada + Dice) com peso automático pra classe daninha (rara). AdamW + warmup + cosine 25 épocas, early stop patience=7 em IoU de validação.
5. Predição streaming num raster GeoTIFF + `recall_por_mancha` (poligonal, não só bloco).

**Pré-requisitos no Drive**:
- `MyDrive/Datasets/DaninhasTreinoClientes/{Giasa,DoisRiosFlaviano,Flaviano01,CelsoSTE2,Celso01}/{imagem,daninhas,plantacao}` — layout já em uso.
- Secret `HF_TOKEN` no Colab (ícone chave) — o checkpoint `dinov3-vitl16-pretrain-sat493m` é *gated*; aceitar termos em `huggingface.co/facebook/dinov3-vitl16-pretrain-sat493m`.

Runtime → GPU **A100** (T4 funciona mas o forward do ViT-L num tile 512² fica na casa dos segundos).

## 1. Montar Drive e instalar dependências

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install rasterio geopandas scikit-image joblib tqdm scikit-learn shapely transformers torch huggingface_hub

## 2. Imports

In [ ]:
import os
import hashlib
import inspect
import math
import numpy as np
import rasterio
from rasterio import features as rfeatures
from rasterio.windows import Window
from rasterio.windows import bounds as window_bounds
from rasterio.transform import Affine
import geopandas as gpd
from shapely.geometry import box
from shapely.ops import unary_union
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor
import joblib
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

## 3. Login no Hugging Face

O checkpoint `dinov3-vitl16-pretrain-sat493m` é **gated**. Aceite os termos na página do modelo e coloque seu token no *Secrets* do Colab (ícone chave à esquerda) como `HF_TOKEN`. Sem o secret, cai no login interativo.

In [ ]:
from huggingface_hub import login

_hf_token = None
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = os.environ.get('HF_TOKEN')

if _hf_token:
    login(token=_hf_token)
    print('Login no Hugging Face OK (via HF_TOKEN).')
else:
    # fallback interativo (cola o token na caixa que aparecer)
    login()

## 4. Configuração

Filosofia do ciclo: **NADA aqui muda entre rodadas** — a mesma config define o instrumento fixo. Só `CICLO` incrementa a cada rodada, pra versionar os artefatos de saída (`_c1.pt`, `_c2.pt`, ...).

Split treino/val é uma **faixa espacial dentro de CADA imagem** (não leave-one-farm-out): `VAL_FRACTION` da largura vira val, com `MARGIN_TILES` de gap descartado no meio pra evitar vazamento (DINOv3 usa contexto de vizinhança). Todas as fazendas contribuem pra treino E val.

In [ ]:
# ---- ciclo (versiona artefatos, não altera treino) ----
CICLO = 1

BASE = '/content/drive/MyDrive/Datasets/DaninhasTreinoClientes'

PARES_CONFIG = [
    {
        'nome'    : 'Giasa',
        'imagem'  : f'{BASE}/Giasa/imagem/Giasa.tif',
        'geojsons': [
            f'{BASE}/Giasa/daninhas/FolhaLargaGiasa (1).geojson',
            f'{BASE}/Giasa/daninhas/FolhaEstreitaGiasa (1).geojson',
            f'{BASE}/Giasa/daninhas/MamonasGiasa (1).geojson',
        ],
        'plantacao': f'{BASE}/Giasa/plantacao/Giasa_plantacao.geojson',
    },
    {
        'nome'    : 'DoisRiosFlaviano',
        'imagem'  : f'{BASE}/DoisRiosFlaviano/imagem/DoisRiosFlaviano.tif',
        'geojsons': [
            f'{BASE}/DoisRiosFlaviano/daninhas/FolhaLargaDoisRiosFlaviano.geojson',
            f'{BASE}/DoisRiosFlaviano/daninhas/FolhaEstreitaDoisRiosFlaviano.geojson',
            f'{BASE}/DoisRiosFlaviano/daninhas/MamonasDoisRiosFlaviano.geojson',
        ],
        'plantacao': f'{BASE}/DoisRiosFlaviano/plantacao/DoisRiosFlaviano_plantacao.geojson',
    },
    {
        'nome'    : 'Flaviano',
        'imagem'  : f'{BASE}/Flaviano01/imagem/Flaviano01.tif',
        'geojsons': [
            f'{BASE}/Flaviano01/daninhas/FolhaLargaFlaviano-1 (1).geojson',
            f'{BASE}/Flaviano01/daninhas/FolhaEstreitaFlaviano-1 (1).geojson',
            f'{BASE}/Flaviano01/daninhas/MamonasFlaviano-1 (1).geojson',
        ],
        'plantacao': f'{BASE}/Flaviano01/plantacao/Flaviano_plantacao.geojson',
    },
    {
        'nome'    : 'CelsoSTE2',
        'imagem'  : f'{BASE}/CelsoSTE2/imagem/CelsoSTE2.tif',
        'geojsons': [
            f'{BASE}/CelsoSTE2/daninhas/FolhaLargaCelsoSTE-2 (1).geojson',
            f'{BASE}/CelsoSTE2/daninhas/FolhaEstreitaCelsoSTE-2 (1).geojson',
            f'{BASE}/CelsoSTE2/daninhas/MamonasCelsoSTE-2 (1).geojson',
        ],
        'plantacao': f'{BASE}/CelsoSTE2/plantacao/CelsoSTE2_plantacao.geojson',
    },
    {
        'nome'    : 'Celso01',
        'imagem'  : f'{BASE}/Celso01/imagem/Celso01.tif',
        'geojsons': [
            f'{BASE}/Celso01/daninhas/FolhasLargas_Celso_01 (1).geojson',
            f'{BASE}/Celso01/daninhas/FolhaEstreita_Celso_01 (1).geojson',
        ],
        'plantacao': f'{BASE}/Celso01/plantacao/Celso01_plantacao.geojson',
    },
]

# So o Flaviano tem o poligono de 'plantacao' com semantica invertida (marca a
# area SEM interesse). Confirmado na sessao do notebook base.
INVERTED_PLANTACAO = {'Flaviano'}

# ---- DINOv3 (variante satelite) ----
DINO_MODEL = 'facebook/dinov3-vitl16-pretrain-sat493m'
TILE = 512          # tamanho do tile enviado ao DINOv3 (multiplo de PATCH)
PATCH = 16          # patch_size do DINOv3 == block_size (1 patch token = 1 bloco)

# ---- Amostragem de tiles (geometria, sem GPU) ----
MAX_WEED_TILES_PER_SITE = 100000   # ~sem teto: pega TODA a daninha (classe escassa)
MAX_CULTIVO_TILES_PER_SITE = 1800  # tiles de cultivo (amostra aleatoria ate esse teto)

# ---- Split treino/validacao: faixa espacial dentro de CADA imagem ----
VAL_FRACTION = 0.20   # fracao (da extremidade 'de cima' do eixo) que vira validacao
MARGIN_TILES = 1      # faixa-gap descartada entre treino e validacao (em tiles)
SPLIT_AXIS = 'x'      # 'x' divide por coluna de tile; 'y' divide por linha

# ---- Rotulo por bloco ----
BLOCK_SIZE = PATCH
DANINHA_MIN_FRAC = 0.4   # fracao minima de pixels-daninha pro bloco virar daninha
MIN_NODATA_FRACTION = 0.5
NODATA_THRESHOLD = 2
IGNORE_LABEL = 255
CLASS_NAMES = {0: 'cultivo', 1: 'daninha'}

# ---- Transformer (head treinavel sobre os patch tokens de um tile) ----
TRANSFORMER_LAYERS = 2
NHEAD = 8              # 1024 / 8 = 128 por cabeca
TRANSFORMER_DROPOUT = 0.15
CLASSIFIER_DROPOUT = 0.2
EPOCHS = 25
LR = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
TILE_BATCH = 8         # tiles por batch (cada tile = 1024 tokens)
DANINHA_CLASS_WEIGHT = None   # None -> calculado automaticamente pela frequencia no treino
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 7
RANDOM_STATE = 42

# ---- Loss: cross-entropy (padrao) ou Tversky (pro-recall) ----
LOSS_TYPE = 'ce_dice'    # 'ce' | 'tversky' | 'ce_dice' | 'ce_tversky'
TVERSKY_ALPHA = 0.3      # peso do falso positivo
TVERSKY_BETA = 0.7       # peso do falso negativo -- beta > alpha favorece RECALL
TVERSKY_SMOOTH = 1.0
DICE_LAMBDA = 1.0

# ---- Saidas versionadas por ciclo ----
OUTPUT_MODEL = f'/content/drive/MyDrive/modelo_transformer_dinov3_clientes_c{CICLO}.pt'
REPORT_PATH  = f'/content/drive/MyDrive/transformer_dinov3_relatorio_clientes_c{CICLO}.txt'
# Checkpoint das features do DINOv3: NAO depende do ciclo (mesmo dataset -> mesmas features).
# Se voce anotar/mudar geojsons entre ciclos, a assinatura muda e recalcula automaticamente.
CHECKPOINT_DIR = '/content/drive/MyDrive/transformer_dinov3_checkpoint_clientes'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Ciclo:', CICLO, '| output:', OUTPUT_MODEL)

## 5. Carregar DINOv3 (congelado) + `extract_tile_features`

Backbone congelado (`eval`, `no_grad`, na GPU). Normalização lida do `AutoImageProcessor` do checkpoint satélite (`mean=[0.430,0.411,0.296]`, `std=[0.213,0.156,0.143]`) — norma errada degrada as features em silêncio. Descarta CLS + register tokens antes do reshape pra grade.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE, (torch.cuda.get_device_name(0) if DEVICE == 'cuda' else ''))

# Normalizacao: carregada DO MODELO (NAO ImageNet).
proc = AutoImageProcessor.from_pretrained(DINO_MODEL)
DINOV3_MEAN = [float(x) for x in proc.image_mean]
DINOV3_STD  = [float(x) for x in proc.image_std]
print('mean:', DINOV3_MEAN, ' std:', DINOV3_STD)

print('Carregando modelo DINOv3 (congelado):', DINO_MODEL)
model = AutoModel.from_pretrained(DINO_MODEL)
model.eval()
for p in model.parameters():
    p.requires_grad = False   # so o head (transformer) treina
model = model.to(DEVICE)

PATCH_SIZE = model.config.patch_size
NUM_REGISTER_TOKENS = getattr(model.config, 'num_register_tokens', 0)
HIDDEN_SIZE = model.config.hidden_size
print(f'patch_size={PATCH_SIZE} hidden_size={HIDDEN_SIZE} num_register_tokens={NUM_REGISTER_TOKENS}')
assert PATCH_SIZE == PATCH, f'PATCH ({PATCH}) != patch_size do modelo ({PATCH_SIZE})'

mean_t = torch.tensor(DINOV3_MEAN, device=DEVICE).view(1, 3, 1, 1)
std_t  = torch.tensor(DINOV3_STD,  device=DEVICE).view(1, 3, 1, 1)
SUPPORTS_INTERP = 'interpolate_pos_encoding' in inspect.signature(model.forward).parameters

In [ ]:
_TOKENS_PRINTED = {'done': False}

def extract_tile_features(tile_uint8_hw3):
    """tile (H,W,3) uint8, H e W multiplos de PATCH_SIZE.
    Retorna (H/PATCH, W/PATCH, HIDDEN_SIZE) float32 -- 1 vetor por bloco/patch."""
    h, w, _ = tile_uint8_hw3.shape
    t = torch.from_numpy(tile_uint8_hw3.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0)
    t = t.to(DEVICE)
    t = (t - mean_t) / std_t
    kwargs = {'interpolate_pos_encoding': True} if SUPPORTS_INTERP else {}
    with torch.no_grad():
        out = model(pixel_values=t, **kwargs)
    tokens = out.last_hidden_state.float().cpu()[0]   # (1 + registers + n_patches, hidden)
    # ARMADILHA: descartar CLS + register tokens ANTES do reshape pra grade.
    patch_tokens = tokens[1 + NUM_REGISTER_TOKENS:]
    gh, gw = h // PATCH_SIZE, w // PATCH_SIZE
    expected = gh * gw
    if patch_tokens.shape[0] != expected:
        raise RuntimeError(
            f'Esperava {expected} patch tokens (grade {gh}x{gw}), vieram {patch_tokens.shape[0]}. '
            f'(total de tokens={tokens.shape[0]}, descartados={1 + NUM_REGISTER_TOKENS})')
    if not _TOKENS_PRINTED['done']:
        print(f'  tokens totais={tokens.shape[0]} | descartados (CLS+{NUM_REGISTER_TOKENS} reg)='
              f'{1 + NUM_REGISTER_TOKENS} | patch tokens={patch_tokens.shape[0]} | grade {gh}x{gw}')
        _TOKENS_PRINTED['done'] = True
    return patch_tokens.reshape(gh, gw, -1).numpy()


def nodata_valid_mask(tile_rgb_hw3, gh, gw):
    """(gh, gw) bool -- bloco valido se < MIN_NODATA_FRACTION dos pixels for nodata."""
    nodata_mask = np.all(tile_rgb_hw3 <= NODATA_THRESHOLD, axis=-1)
    nd_blk = nodata_mask.reshape(gh, PATCH_SIZE, gw, PATCH_SIZE)
    nodata_frac = nd_blk.mean(axis=(1, 3))
    return nodata_frac < MIN_NODATA_FRACTION

## 6. Rótulos (rasterizar daninha + plantação) e rótulo por bloco

In [ ]:
def load_geoms(paths, raster_crs):
    geoms = []
    for p in paths:
        if not os.path.exists(p):
            print(f'  aviso: arquivo nao encontrado, pulando: {p}')
            continue
        try:
            gdf = gpd.read_file(p)
        except Exception as e:
            print(f'  aviso: nao consegui ler {p}: {e}')
            continue
        if gdf.crs is not None and gdf.crs != raster_crs:
            gdf = gdf.to_crs(raster_crs)
        for g in gdf.geometry:
            if g is None or g.is_empty:
                continue
            if not g.is_valid:
                g = g.buffer(0)
            if g.is_empty:
                continue
            geoms.append(g)
    return geoms


def rasterize_labels(daninha_geoms, plantacao_geoms, win_transform, height, width, invert_plantacao=False):
    """invert_plantacao=True (so Flaviano): area valida = COMPLEMENTO do poligono de plantacao.
    Sem poligono de plantacao -> nada e' considerado valido (mais seguro que assumir a imagem inteira)."""
    if plantacao_geoms:
        plant_mask = rfeatures.rasterize(
            [(g, 1) for g in plantacao_geoms], out_shape=(height, width),
            transform=win_transform, fill=0, dtype=np.uint8,
        ).astype(bool)
        if invert_plantacao:
            plant_mask = ~plant_mask
    else:
        plant_mask = np.zeros((height, width), dtype=bool)
    if daninha_geoms:
        daninha_mask = rfeatures.rasterize(
            [(g, 1) for g in daninha_geoms], out_shape=(height, width),
            transform=win_transform, fill=0, dtype=np.uint8,
        ).astype(bool)
    else:
        daninha_mask = np.zeros((height, width), dtype=bool)
    label = np.full((height, width), IGNORE_LABEL, dtype=np.uint8)
    label[plant_mask] = 0                        # cultivo
    label[plant_mask & daninha_mask] = 1         # daninha tem prioridade
    return label


def block_labels_from_pixels(label, gh, gw):
    """Reduz o label por pixel (TILE x TILE) pra grade de blocos (gh x gw).
    DANINHA_MIN_FRAC controla a fracao minima de pixels-daninha; prioridade daninha."""
    label_blk = label[:gh * PATCH, :gw * PATCH].reshape(gh, PATCH, gw, PATCH)
    n_daninha = (label_blk == 1).sum(axis=(1, 3))
    n_cultivo = (label_blk == 0).sum(axis=(1, 3))
    n_valid_safe = np.maximum(n_daninha + n_cultivo, 1)
    has_cultivo = n_cultivo > 0
    if DANINHA_MIN_FRAC > 0:
        is_daninha = (n_daninha / n_valid_safe) >= DANINHA_MIN_FRAC
    else:
        is_daninha = n_daninha > 0
    block_label = np.full((gh, gw), IGNORE_LABEL, dtype=np.uint8)
    block_label[has_cultivo] = 0
    block_label[is_daninha] = 1                 # prioridade sobre cultivo
    return block_label

## 7. Amostragem de tiles (geometria, sem GPU)

Classifica a grade de tiles de `TILE` px só com `gdf.sindex` sobre os bounds de cada tile: tile que toca daninha -> daninha; tile na plantação sem daninha -> cultivo (amostrado); fora da plantação -> descartado.

In [ ]:
def select_tiles(src, daninha_geoms, plantacao_geoms, invert_plantacao, rng):
    W, H = src.width, src.height
    tiles_x = W // TILE
    tiles_y = H // TILE

    daninha_gs = gpd.GeoSeries(daninha_geoms) if daninha_geoms else None
    d_sindex = daninha_gs.sindex if daninha_gs is not None else None

    if plantacao_geoms:
        plant_union = unary_union([g.buffer(0) for g in plantacao_geoms])
        if plant_union.is_empty:
            plant_union = None
    else:
        plant_union = None

    weed_pos = []
    cultivo_candidates = []
    for ty in range(tiles_y):
        for tx in range(tiles_x):
            col_off = tx * TILE
            row_off = ty * TILE
            win = Window(col_off, row_off, TILE, TILE)
            left, bottom, right, top = window_bounds(win, src.transform)
            tbox = box(left, bottom, right, top)

            is_weed = False
            if d_sindex is not None:
                hits = d_sindex.query(tbox, predicate='intersects')
                is_weed = len(hits) > 0
            if is_weed:
                weed_pos.append((col_off, row_off))
                continue

            if plant_union is not None:
                if invert_plantacao:
                    # area valida = FORA do poligono; candidato se nao totalmente dentro
                    cand = not plant_union.contains(tbox)
                else:
                    cand = plant_union.intersects(tbox)
                if cand:
                    cultivo_candidates.append((col_off, row_off))

    if len(weed_pos) > MAX_WEED_TILES_PER_SITE:
        idx = rng.choice(len(weed_pos), size=MAX_WEED_TILES_PER_SITE, replace=False)
        weed_pos = [weed_pos[i] for i in sorted(idx)]
    if len(cultivo_candidates) > MAX_CULTIVO_TILES_PER_SITE:
        idx = rng.choice(len(cultivo_candidates), size=MAX_CULTIVO_TILES_PER_SITE, replace=False)
        cultivo_pos = [cultivo_candidates[i] for i in sorted(idx)]
    else:
        cultivo_pos = cultivo_candidates
    return weed_pos, cultivo_pos

## 8. Coleta por TILE (com checkpoint)

Guarda por tile: `tokens (1024, 1024) float16` + `labels (1024,) uint8` + `positions (col_off, row_off)`. Checkpoint por site em `CHECKPOINT_DIR` (assinatura só dos params que afetam a COLETA — mudar `VAL_FRACTION`/`MARGIN_TILES`/`SPLIT_AXIS` NÃO invalida). Se o Colab cair, rode de novo — sites já processados voltam do checkpoint.

In [ ]:
def collect_site_tiles(nome, imagem_path, daninha_paths, plantacao_path):
    """Retorna (tokens_list, labels_list, positions_list, n_tiles_x, n_tiles_y)."""
    rng = np.random.default_rng(RANDOM_STATE)
    invert_plantacao = nome in INVERTED_PLANTACAO
    if invert_plantacao:
        print(f'  aviso: {nome} em INVERTED_PLANTACAO -- area valida = FORA do poligono')

    tokens_list, labels_list, positions_list = [], [], []
    with rasterio.open(imagem_path) as src:
        raster_crs = src.crs
        daninha_geoms = load_geoms(daninha_paths, raster_crs)
        plantacao_geoms = load_geoms([plantacao_path], raster_crs) if plantacao_path else []

        n_tiles_x = src.width // TILE
        n_tiles_y = src.height // TILE

        weed_pos, cultivo_pos = select_tiles(src, daninha_geoms, plantacao_geoms, invert_plantacao, rng)
        print(f'  tiles selecionados: {len(weed_pos)} de daninha + {len(cultivo_pos)} de cultivo')
        all_tiles = weed_pos + cultivo_pos
        gh = gw = TILE // PATCH

        for (col_off, row_off) in tqdm(all_tiles, desc=nome):
            win = Window(col_off, row_off, TILE, TILE)
            tile = np.moveaxis(src.read([1, 2, 3], window=win, boundless=True, fill_value=0), 0, -1)
            feats = extract_tile_features(tile)                # (gh, gw, HIDDEN_SIZE)
            valid = nodata_valid_mask(tile, gh, gw)             # (gh, gw)
            win_transform = src.window_transform(win)
            label = rasterize_labels(daninha_geoms, plantacao_geoms, win_transform, TILE, TILE, invert_plantacao)
            block_label = block_labels_from_pixels(label, gh, gw)
            block_label = np.where(valid, block_label, IGNORE_LABEL).astype(np.uint8)

            if np.all(block_label == IGNORE_LABEL):
                del tile, feats
                continue

            tokens_list.append(feats.reshape(-1, HIDDEN_SIZE).astype(np.float16))
            labels_list.append(block_label.reshape(-1))
            positions_list.append((col_off, row_off))
            del tile, feats
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    n_daninha = sum(int((l == 1).sum()) for l in labels_list)
    n_cultivo = sum(int((l == 0).sum()) for l in labels_list)
    print(f'  tiles validos: {len(tokens_list)} | blocos cultivo={n_cultivo} daninha={n_daninha}')
    return tokens_list, labels_list, positions_list, n_tiles_x, n_tiles_y


def config_signature():
    """Assinatura dos parametros que afetam a COLETA (NAO inclui VAL_FRACTION/MARGIN_TILES/SPLIT_AXIS)."""
    payload = repr((
        'dinov3-transformer', DINO_MODEL, TILE, PATCH, DINOV3_MEAN, DINOV3_STD,
        DANINHA_MIN_FRAC, MIN_NODATA_FRACTION, NODATA_THRESHOLD,
        MAX_WEED_TILES_PER_SITE, MAX_CULTIVO_TILES_PER_SITE,
    ))
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def collect_site_tiles_cached(cfg):
    sig = config_signature()
    ck_path = f"{CHECKPOINT_DIR}/{cfg['nome']}_{sig}.joblib"
    if os.path.exists(ck_path):
        d = joblib.load(ck_path)
        print(f"  (checkpoint encontrado -- pulando reprocessamento, {len(d['labels'])} tiles)")
        return d['tokens'], d['labels'], d['positions'], d['n_tiles_x'], d['n_tiles_y']
    tokens_list, labels_list, positions_list, n_tiles_x, n_tiles_y = collect_site_tiles(
        cfg['nome'], cfg['imagem'], cfg['geojsons'], cfg['plantacao'])
    joblib.dump({
        'tokens': tokens_list, 'labels': labels_list, 'positions': positions_list,
        'n_tiles_x': n_tiles_x, 'n_tiles_y': n_tiles_y,
    }, ck_path)
    return tokens_list, labels_list, positions_list, n_tiles_x, n_tiles_y

## 9. Rodar a coleta

Coleta tiles de TODOS os sites (não há mais distinção treino/validação nesta etapa — é decidida geometricamente na seção 10). Se o Colab desconectar, rode de novo — sites já processados vêm do checkpoint.

In [ ]:
site_data = {}
for cfg in PARES_CONFIG:
    print(f"\n=== {cfg['nome']} ===")
    tokens, labels, positions, n_tiles_x, n_tiles_y = collect_site_tiles_cached(cfg)
    site_data[cfg['nome']] = {
        'tokens': tokens, 'labels': labels, 'positions': positions,
        'n_tiles_x': n_tiles_x, 'n_tiles_y': n_tiles_y,
    }

n_total_tiles = sum(len(d['tokens']) for d in site_data.values())
print(f'\nTotal de tiles coletados (todos os sites): {n_total_tiles}')
if n_total_tiles == 0:
    raise RuntimeError('Nenhum tile coletado -- confira os caminhos no Drive.')

## 10. Split treino/validação (faixa espacial dentro da AOI de cada imagem)

Corte medido pelos TILES COLETADOS de cada site (AOI = daninha + cultivo dentro da plantação já filtrados), NÃO pela largura/altura bruta do raster. Se a plantação estiver concentrada num canto, medir pela largura toda deixaria a faixa de validação vazia. Todas as fazendas contribuem pra treino E val.

In [ ]:
def split_train_val(site_data, val_fraction, margin_tiles, axis='x'):
    if axis not in ('x', 'y'):
        raise ValueError(f"SPLIT_AXIS invalido: {axis!r} (use 'x' ou 'y')")

    tokens_train, labels_train = [], []
    tokens_val, labels_val = [], []

    for nome, d in site_data.items():
        items = list(zip(d['tokens'], d['labels'], d['positions']))
        total = len(items)
        if total == 0:
            print(f'  {nome}: 0 tiles coletados -- pulando.')
            continue

        def tile_idx(pos, axis=axis):
            col_off, row_off = pos
            return (col_off // TILE) if axis == 'x' else (row_off // TILE)

        counts = {}
        for _, _, pos in items:
            idx = tile_idx(pos)
            counts[idx] = counts.get(idx, 0) + 1

        if val_fraction <= 0:
            val_start = max(counts) + 1   # nenhum tile satisfaz idx >= val_start -> val vazio
        else:
            accumulated = 0
            val_start = min(counts)      # fallback: val_fraction > 1 -> tudo vira val
            for idx in sorted(counts.keys(), reverse=True):
                accumulated += counts[idx]
                val_start = idx
                if accumulated / total >= val_fraction:
                    break

        train_end = val_start - margin_tiles
        n_tr = n_va = n_gap = 0
        for tokens, labels, pos in items:
            idx = tile_idx(pos)
            if idx >= val_start:
                tokens_val.append(tokens); labels_val.append(labels); n_va += 1
            elif idx < train_end:
                tokens_train.append(tokens); labels_train.append(labels); n_tr += 1
            else:
                n_gap += 1
        val_frac_alcancada = n_va / total
        print(f'  {nome}: treino={n_tr} val={n_va} descartado(faixa-gap)={n_gap} '
              f'| total_aoi={total} val_start({axis})={val_start} margin={margin_tiles} '
              f'| fracao val alcancada={val_frac_alcancada:.3f} (alvo={val_fraction:.3f})')
        if n_tr == 0:
            print(f'  aviso: {nome} ficou SEM tiles de treino apos o split!')
        if n_va == 0:
            print(f'  aviso: {nome} ficou SEM tiles de validacao apos o split!')

    return tokens_train, labels_train, tokens_val, labels_val


tokens_train, labels_train, tokens_val, labels_val = split_train_val(
    site_data, VAL_FRACTION, MARGIN_TILES, SPLIT_AXIS)

print(f'\nTiles de treino: {len(tokens_train)}')
print(f'Tiles de validacao: {len(tokens_val)}')

if len(tokens_train) == 0:
    raise RuntimeError('Nenhum tile de treino apos o split -- ajuste VAL_FRACTION/MARGIN_TILES.')

# ---- peso da classe daninha (compensa raridade, ~0.2-0.8% real) ----
if DANINHA_CLASS_WEIGHT is None:
    n_cultivo = sum(int((l == 0).sum()) for l in labels_train)
    n_daninha = sum(int((l == 1).sum()) for l in labels_train)
    n_daninha_safe = max(n_daninha, 1)
    daninha_weight = n_cultivo / n_daninha_safe
    daninha_weight = float(np.clip(daninha_weight, 1.0, 200.0))
else:
    daninha_weight = float(DANINHA_CLASS_WEIGHT)
print(f'peso da classe daninha (auto={DANINHA_CLASS_WEIGHT is None}): {daninha_weight:.2f}')

## 11. Dataset / DataLoader de tiles

Cada item = 1 tile inteiro: `tokens (1024, HIDDEN_SIZE) float32` + `labels (1024,) uint8`. Grade fixa 32×32, sem padding.

In [ ]:
N_TOKENS = (TILE // PATCH) * (TILE // PATCH)   # 32*32 = 1024

class TileTokenDataset(Dataset):
    def __init__(self, tokens_list, labels_list):
        self.tokens_list = tokens_list
        self.labels_list = labels_list

    def __len__(self):
        return len(self.tokens_list)

    def __getitem__(self, idx):
        tokens = torch.from_numpy(self.tokens_list[idx].astype(np.float32))   # (N_TOKENS, HIDDEN_SIZE)
        labels = torch.from_numpy(self.labels_list[idx].astype(np.int64))     # (N_TOKENS,)
        return tokens, labels


def collate_tiles(batch):
    tokens = torch.stack([b[0] for b in batch], dim=0)   # (B, N_TOKENS, HIDDEN_SIZE)
    labels = torch.stack([b[1] for b in batch], dim=0)   # (B, N_TOKENS)
    return tokens, labels


train_loader = DataLoader(
    TileTokenDataset(tokens_train, labels_train),
    batch_size=TILE_BATCH, shuffle=True, collate_fn=collate_tiles, drop_last=False,
)
val_loader = None
if len(tokens_val) > 0:
    val_loader = DataLoader(
        TileTokenDataset(tokens_val, labels_val),
        batch_size=TILE_BATCH, shuffle=False, collate_fn=collate_tiles, drop_last=False,
    )
print(f'batches de treino: {len(train_loader)}' + (f' | batches de validacao: {len(val_loader)}' if val_loader else ''))

## 12. Arquitetura do head — transformer curto sobre os patch tokens

```
DINOv3 (congelado) -> tokens (B, 1024, 1024)
  -> LayerNorm de entrada -> + positional encoding aprendivel (1, 1024, 1024)
    -> TRANSFORMER_LAYERS x TransformerEncoderLayer (d_model=1024, nhead=NHEAD,
        batch_first=True, norm_first=True, activation='gelu')
      -> LayerNorm -> Dropout(CLASSIFIER_DROPOUT) -> Linear(1024, 2)   # logit POR TOKEN
```

Cada token = 1 bloco de 16px. Saída: `(B, 1024, 2)`. Só o head treina; DINOv3 fica sempre congelado.

In [ ]:
class TileTransformerHead(nn.Module):
    def __init__(self, hidden_size, n_tokens, n_layers=2, nhead=8, dropout=0.1,
                 classifier_dropout=0.0, n_classes=2):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.zeros(1, n_tokens, hidden_size))
        nn.init.trunc_normal_(self.pos_encoding, std=0.02)
        # Os tokens crus do DINOv3 tem escala bem maior que o pos_encoding (std=0.02) --
        # sem normalizar, o sinal de posicao fica diluido. Normaliza os tokens ANTES.
        self.input_norm = nn.LayerNorm(hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=nhead, dim_feedforward=hidden_size * 2,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(hidden_size)
        self.class_dropout = nn.Dropout(classifier_dropout)   # opcional, 0.0 = no-op
        self.classifier = nn.Linear(hidden_size, n_classes)

    def forward(self, tokens):
        # tokens: (B, N_TOKENS, HIDDEN_SIZE)
        tokens_norm = self.input_norm(tokens)
        x = tokens_norm + self.pos_encoding
        x = self.encoder(x)
        x = self.norm(x)
        x = self.class_dropout(x)
        return self.classifier(x)   # (B, N_TOKENS, n_classes)


head = TileTransformerHead(
    hidden_size=HIDDEN_SIZE, n_tokens=N_TOKENS, n_layers=TRANSFORMER_LAYERS,
    nhead=NHEAD, dropout=TRANSFORMER_DROPOUT, classifier_dropout=CLASSIFIER_DROPOUT,
    n_classes=2,
).to(DEVICE)
n_params = sum(p.numel() for p in head.parameters() if p.requires_grad)
print(f'Head transformer: {n_params:,} parametros treinaveis')

## 13. Loss, otimizador, scheduler e loop de treino

In [ ]:
def tversky_loss_daninha(logits, labels, alpha, beta, smooth):
    """Tversky soft focada em daninha (1), mascarando IGNORE_LABEL. beta > alpha favorece recall."""
    valid = (labels != IGNORE_LABEL)
    proba = torch.softmax(logits, dim=-1)[..., 1]     # (B, N) proba da classe daninha
    g1 = (labels == 1).float()
    p1 = torch.where(valid, proba, torch.zeros_like(proba))
    g1 = torch.where(valid, g1, torch.zeros_like(g1))
    tp = (p1 * g1).sum()
    fp = (p1 * (1.0 - g1) * valid.float()).sum()
    fn = ((1.0 - p1) * g1).sum()
    tversky = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
    return 1.0 - tversky


def dice_loss_daninha(logits, labels, smooth):
    valid = (labels != IGNORE_LABEL)
    proba = torch.softmax(logits, dim=-1)[..., 1]
    g1 = (labels == 1).float()
    p1 = torch.where(valid, proba, torch.zeros_like(proba))
    g1 = torch.where(valid, g1, torch.zeros_like(g1))
    inter = (p1 * g1).sum()
    dice = (2.0 * inter + smooth) / (p1.sum() + g1.sum() + smooth)
    return 1.0 - dice


ce_loss_fn = nn.CrossEntropyLoss(
    ignore_index=IGNORE_LABEL,
    weight=torch.tensor([1.0, daninha_weight], dtype=torch.float32, device=DEVICE),
)

def compute_loss(logits, labels):
    if LOSS_TYPE == 'ce':
        return ce_loss_fn(logits.reshape(-1, 2), labels.reshape(-1))
    elif LOSS_TYPE == 'tversky':
        return tversky_loss_daninha(logits, labels, TVERSKY_ALPHA, TVERSKY_BETA, TVERSKY_SMOOTH)
    elif LOSS_TYPE == 'ce_dice':
        ce_term = ce_loss_fn(logits.reshape(-1, 2), labels.reshape(-1))
        return ce_term + DICE_LAMBDA * dice_loss_daninha(logits, labels, TVERSKY_SMOOTH)
    elif LOSS_TYPE == 'ce_tversky':
        ce_term = ce_loss_fn(logits.reshape(-1, 2), labels.reshape(-1))
        return ce_term + DICE_LAMBDA * tversky_loss_daninha(logits, labels, TVERSKY_ALPHA, TVERSKY_BETA, TVERSKY_SMOOTH)
    else:
        raise ValueError(f"LOSS_TYPE invalido: {LOSS_TYPE!r}")

print(f'LOSS_TYPE={LOSS_TYPE!r}')

optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch_idx):
    # epoch_idx e' 0-based (chamado por LambdaLR antes/durante cada step()).
    if WARMUP_EPOCHS > 0 and epoch_idx < WARMUP_EPOCHS:
        return (epoch_idx + 1) / WARMUP_EPOCHS
    denom = max(EPOCHS - WARMUP_EPOCHS, 1)
    progress = (epoch_idx - WARMUP_EPOCHS) / denom
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def run_epoch(loader, train=True):
    head.train(mode=train)
    total_loss, n_batches = 0.0, 0
    for tokens, labels in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = head(tokens)                        # (B, N_TOKENS, 2)
            loss = compute_loss(logits, labels)
        if train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        total_loss += float(loss.item())
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate_iou(loader):
    """IoU da classe daninha (mascarando IGNORE_LABEL) -- usado pro early stop."""
    if loader is None:
        return None, None, None
    head.eval()
    tp = fp = fn = 0
    all_true, all_pred = [], []
    for tokens, labels in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        logits = head(tokens)                             # (B, N_TOKENS, 2)
        pred = logits.argmax(dim=-1).cpu().numpy().reshape(-1)
        true = labels.numpy().reshape(-1)
        valid = true != IGNORE_LABEL
        pred_v, true_v = pred[valid], true[valid]
        tp += int(np.sum((pred_v == 1) & (true_v == 1)))
        fp += int(np.sum((pred_v == 1) & (true_v == 0)))
        fn += int(np.sum((pred_v == 0) & (true_v == 1)))
        all_true.append(true_v); all_pred.append(pred_v)
    iou = tp / max(tp + fp + fn, 1)
    return iou, (np.concatenate(all_true) if all_true else np.array([])), (np.concatenate(all_pred) if all_pred else np.array([]))

In [ ]:
best_iou = -1.0
best_state = None
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_iou, _, _ = evaluate_iou(val_loader)
    current_lr = optimizer.param_groups[0]['lr']
    msg = f'epoca {epoch}/{EPOCHS} | LR: {current_lr:.2e} | loss treino: {train_loss:.4f}'
    if val_iou is not None:
        msg += f' | IoU daninha (val): {val_iou:.4f}'
        if val_iou > best_iou:
            best_iou = val_iou
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            epochs_no_improve = 0
            msg += '  (melhor ate agora, salvando estado)'
        else:
            epochs_no_improve += 1
    else:
        # sem validacao -- guarda sempre o estado mais recente
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
    print(msg)
    scheduler.step()   # por epoca
    if val_loader is not None and epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f'early stop -- sem melhora no IoU de validacao ha {EARLY_STOP_PATIENCE} epocas.')
        break

if best_state is not None:
    head.load_state_dict(best_state)
print('Treino concluido.' + (f' Melhor IoU (val): {best_iou:.4f}' if best_iou >= 0 else ''))

# Salva ANTES de imprimir metricas (resiste a queda de sessao).
torch.save({
    'head_state_dict': head.state_dict(),
    'hidden_size': HIDDEN_SIZE,
    'n_tokens': N_TOKENS,
    'transformer_layers': TRANSFORMER_LAYERS,
    'nhead': NHEAD,
    'transformer_dropout': TRANSFORMER_DROPOUT,
    'classifier_dropout': CLASSIFIER_DROPOUT,
    'loss_type': LOSS_TYPE,
    'tversky_alpha': TVERSKY_ALPHA,
    'tversky_beta': TVERSKY_BETA,
    'tversky_smooth': TVERSKY_SMOOTH,
    'dice_lambda': DICE_LAMBDA,
    'warmup_epochs': WARMUP_EPOCHS,
    'dino_model': DINO_MODEL,
    'dinov3_mean': DINOV3_MEAN,
    'dinov3_std': DINOV3_STD,
    'block_size': PATCH,
    'patch_size': PATCH,
    'tile': TILE,
    'num_register_tokens': NUM_REGISTER_TOKENS,
    'nodata_threshold': NODATA_THRESHOLD,
    'min_nodata_fraction': MIN_NODATA_FRACTION,
    'daninha_class_weight': daninha_weight,
    'class_names': CLASS_NAMES,
    'best_val_iou': best_iou,
    'ciclo': CICLO,
}, OUTPUT_MODEL)
print(f'Modelo salvo em {OUTPUT_MODEL}')

In [ ]:
# ---- metricas finais na faixa de validacao (holdout espacial, agregada de todas as fazendas) ----
report_lines = []
if val_loader is not None:
    val_iou, y_true, y_pred = evaluate_iou(val_loader)
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    report_lines.append(
        f'=== Validacao (holdout espacial: {VAL_FRACTION:.0%} da faixa "{SPLIT_AXIS}", '
        f'margem={MARGIN_TILES} tiles, agregado de todas as fazendas) -- Transformer -- ciclo {CICLO} ===')
    report_lines.append(f'IoU daninha: {val_iou:.4f}')
    report_lines.append(f'Precision daninha: {precision:.4f}')
    report_lines.append(f'Recall daninha: {recall:.4f}')
    report_lines.append(classification_report(y_true, y_pred, target_names=['cultivo', 'daninha'], zero_division=0))
    report_lines.append('Matriz de confusao [linhas=verdadeiro, colunas=previsto], ordem [cultivo, daninha]:')
    report_lines.append(str(confusion_matrix(y_true, y_pred, labels=[0, 1])))
else:
    report_lines.append('Sem tiles de validacao apos o split (VAL_FRACTION=0?) -- pulei o classification_report.')

report_text = '\n'.join(report_lines)
print(report_text)
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report_text)
print(f'\nRelatorio salvo em {REPORT_PATH}')

### Recuperar o modelo sem re-treinar (se a sessão cair)

In [ ]:
# import torch
# bundle = torch.load(OUTPUT_MODEL, map_location='cpu')
# head = TileTransformerHead(
#     hidden_size=bundle['hidden_size'], n_tokens=bundle['n_tokens'],
#     n_layers=bundle['transformer_layers'], nhead=bundle['nhead'],
#     dropout=bundle['transformer_dropout'],
#     classifier_dropout=bundle.get('classifier_dropout', 0.0), n_classes=2,
# )
# head.load_state_dict(bundle['head_state_dict'])
# head.eval()
# print('IoU (val) salvo no bundle:', bundle['best_val_iou'])

## 14. Predição numa imagem inteira (streaming por tiles)

Aplica DINOv3 -> transformer -> argmax tile a tile (RAM constante), gerando um raster em resolução de bloco (`out_transform = base_transform * Affine.scale(PATCH, PATCH)`), georreferenciado, com colormap cultivo/daninha/nodata. `PREDICT_DANINHA_PROBA_THRESHOLD` opcional reduz falso positivo (softmax em vez de argmax puro).

In [ ]:
PREDICT_IMAGE_PATH = PARES_CONFIG[0]['imagem']    # troque pela imagem desejada
PREDICT_OUTPUT_PATH = f'/content/drive/MyDrive/predicao_transformer_blocos_c{CICLO}.tif'
PREDICT_DANINHA_PROBA_THRESHOLD = None            # ex: 0.7 -- ou None pra argmax direto

PREDICT_CLASS_COLORS = {
    0: (34, 139, 34, 255),     # cultivo
    1: (220, 20, 60, 255),     # daninha
    255: (0, 0, 0, 0),         # nodata
}

head.eval()

@torch.no_grad()
def predict_tile_labels(feats_flat, valid_flat, threshold):
    """feats_flat: (N_TOKENS, HIDDEN_SIZE) float32 numpy. Retorna (N_TOKENS,) uint8."""
    labels = np.full(feats_flat.shape[0], 255, dtype=np.uint8)
    if not valid_flat.any():
        return labels
    x = torch.from_numpy(feats_flat).unsqueeze(0).to(DEVICE)      # (1, N_TOKENS, HIDDEN_SIZE)
    logits = head(x)[0]                                            # (N_TOKENS, 2)
    if threshold is not None:
        proba_daninha = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        pred = (proba_daninha >= threshold).astype(np.uint8)
    else:
        pred = logits.argmax(dim=-1).cpu().numpy().astype(np.uint8)
    labels[valid_flat] = pred[valid_flat]
    return labels


print(f'Predizendo em: {PREDICT_IMAGE_PATH}')
gh = gw = TILE // PATCH

with rasterio.open(PREDICT_IMAGE_PATH) as src:
    print(f'{src.width}x{src.height} px | CRS: {src.crs}')
    base_transform = src.transform
    tiles_x = src.width // TILE
    tiles_y = src.height // TILE
    full_gh = tiles_y * gh
    full_gw = tiles_x * gw

    out_transform = base_transform * Affine.scale(PATCH, PATCH)
    profile = {
        'driver': 'GTiff', 'height': full_gh, 'width': full_gw, 'count': 1, 'dtype': 'uint8',
        'crs': src.crs, 'transform': out_transform, 'compress': 'lzw', 'nodata': 255,
    }
    n_cultivo = n_daninha = 0
    with rasterio.open(PREDICT_OUTPUT_PATH, 'w', **profile) as dst:
        for ty in tqdm(range(tiles_y), desc='predizendo (linhas de tile)'):
            for tx in range(tiles_x):
                col_off = tx * TILE; row_off = ty * TILE
                win = Window(col_off, row_off, TILE, TILE)
                tile = np.moveaxis(src.read([1, 2, 3], window=win, boundless=True, fill_value=0), 0, -1)
                feats = extract_tile_features(tile)                # (gh, gw, HIDDEN_SIZE)
                valid = nodata_valid_mask(tile, gh, gw)
                feats_flat = feats.reshape(-1, HIDDEN_SIZE).astype(np.float32)
                valid_flat = valid.reshape(-1)
                labels = predict_tile_labels(feats_flat, valid_flat, PREDICT_DANINHA_PROBA_THRESHOLD)
                label_tile = labels.reshape(gh, gw)
                n_cultivo += int(np.sum(label_tile == 0))
                n_daninha += int(np.sum(label_tile == 1))
                out_win = Window(col_off=tx * gw, row_off=ty * gh, width=gw, height=gh)
                dst.write(label_tile, 1, window=out_win)
                del tile, feats
                if DEVICE == 'cuda':
                    torch.cuda.empty_cache()
        dst.write_colormap(1, PREDICT_CLASS_COLORS)

n_total = n_cultivo + n_daninha
if n_total > 0:
    print(f'cultivo: {n_cultivo} blocos ({100*n_cultivo/n_total:.2f}%)')
    print(f'daninha: {n_daninha} blocos ({100*n_daninha/n_total:.2f}%)')
print(f'Predicao salva em {PREDICT_OUTPUT_PATH} ({full_gh}x{full_gw} blocos de {PATCH}px)')

## 15. Recall por MANCHA (não deixar nenhuma infestação passar batido)

IoU/recall por bloco (seção 13) mede acerto bloco a bloco, mas não responde "o modelo VIU essa mancha, ou passou 100% batida?". Aqui, pra cada polígono de daninha, verifica se pelo menos `min_blocos` blocos preditos como daninha caem dentro dele. Manchas menores que 1 bloco são excluídas.

`restringir_val=True` filtra manchas pra só contar as da faixa de validação (comparável ao IoU/recall por bloco); `restringir_val=False` (padrão abaixo) reporta sobre o talhão INTEIRO.

In [ ]:
def recall_por_mancha(pred_raster_path, daninha_geojson_paths, min_blocos=1,
                     restringir_val=False, val_fraction=VAL_FRACTION, axis=SPLIT_AXIS):
    with rasterio.open(pred_raster_path) as dst:
        pred = dst.read(1)
        pred_transform = dst.transform
        pred_crs = dst.crs
    pred_height, pred_width = pred.shape

    daninha_geoms = load_geoms(daninha_geojson_paths, pred_crs)
    if not daninha_geoms:
        print('  nenhum poligono de daninha carregado -- nada a reportar.')
        return

    valid_mask = pred != 255
    val_start_bloco = None
    if restringir_val:
        counts_per_idx = valid_mask.sum(axis=0) if axis == 'x' else valid_mask.sum(axis=1)
        total_valid = int(counts_per_idx.sum())
        if total_valid == 0:
            print('  aviso: raster de predicao sem blocos validos -- restringir_val sem efeito.')
        else:
            accumulated = 0
            val_start_bloco = 0
            for idx in range(len(counts_per_idx) - 1, -1, -1):
                accumulated += int(counts_per_idx[idx])
                val_start_bloco = idx
                if accumulated / total_valid >= val_fraction:
                    break
            print(f'  RESTRITO A FAIXA DE VALIDACAO: eixo={axis} val_start_bloco={val_start_bloco} '
                  f'(fracao alvo={val_fraction:.3f}) -- mesma regiao do IoU/recall por bloco.')
    else:
        print('  reportando sobre o TALHAO INTEIRO da imagem predita (nao restrito a validacao).')

    bucket_defs = [(0, 5, '<5 blocos'), (5, 20, '5-20 blocos'),
                   (20, 100, '20-100 blocos'), (100, math.inf, '>100 blocos')]
    bucket_counts = {lbl: [0, 0] for (_, _, lbl) in bucket_defs}

    n_avaliadas = n_menor_1_bloco = n_detectadas = 0
    coberturas_detectadas = []
    for geom in daninha_geoms:
        mask = rfeatures.rasterize(
            [(geom, 1)], out_shape=(pred_height, pred_width),
            transform=pred_transform, fill=0, dtype=np.uint8,
        ).astype(bool)

        if restringir_val and val_start_bloco is not None:
            col_mask = np.zeros_like(mask)
            if axis == 'x':
                col_mask[:, val_start_bloco:] = True
            else:
                col_mask[val_start_bloco:, :] = True
            mask = mask & col_mask

        n_blocos_total = int(mask.sum())
        if n_blocos_total == 0:
            n_menor_1_bloco += 1
            continue

        n_avaliadas += 1
        n_daninha_pred = int(((pred == 1) & mask).sum())
        detectado = n_daninha_pred >= min_blocos
        cobertura = n_daninha_pred / max(n_blocos_total, 1)

        for lo, hi, lbl in bucket_defs:
            if lo <= n_blocos_total < hi:
                bucket_counts[lbl][0] += 1
                if detectado:
                    bucket_counts[lbl][1] += 1
                break

        if detectado:
            n_detectadas += 1
            coberturas_detectadas.append(cobertura)

    print(f'\n=== Recall por mancha ({"faixa de validacao" if restringir_val else "talhao inteiro"}) ===')
    print(f'  manchas com < 1 bloco na area considerada (excluidas da contagem): {n_menor_1_bloco}')
    print(f'  total de manchas avaliadas: {n_avaliadas}')
    if n_avaliadas > 0:
        recall_mancha = n_detectadas / n_avaliadas
        print(f'  detectadas (>= {min_blocos} bloco(s) predito(s) daninha dentro): {n_detectadas}')
        print(f'  recall por mancha: {recall_mancha:.3f}')
    else:
        print('  nenhuma mancha com blocos validos -- recall por mancha indefinido.')

    print('\n  por tamanho de mancha:')
    for _, _, lbl in bucket_defs:
        total_b, det_b = bucket_counts[lbl]
        if total_b == 0:
            print(f'    {lbl}: 0 manchas')
            continue
        print(f'    {lbl}: {det_b}/{total_b} detectadas (recall={det_b/total_b:.3f})')

    if coberturas_detectadas:
        cov = np.array(coberturas_detectadas)
        print('\n  cobertura (fracao de blocos-daninha dentro da mancha) das manchas DETECTADAS:')
        print(f'    media={cov.mean():.3f}  mediana={np.median(cov):.3f}  min={cov.min():.3f}  max={cov.max():.3f}')
    else:
        print('\n  nenhuma mancha detectada -- sem distribuicao de cobertura.')

In [ ]:
# Rodar o recall por mancha na imagem predita da celula anterior.
PATCH_RECALL_MIN_BLOCOS = 1
PATCH_RECALL_RESTRINGIR_VAL = False
PATCH_RECALL_DANINHA_GEOJSONS = PARES_CONFIG[0]['geojsons']   # geojsons da imagem predita

recall_por_mancha(
    PREDICT_OUTPUT_PATH, PATCH_RECALL_DANINHA_GEOJSONS,
    min_blocos=PATCH_RECALL_MIN_BLOCOS, restringir_val=PATCH_RECALL_RESTRINGIR_VAL,
    val_fraction=VAL_FRACTION, axis=SPLIT_AXIS,
)

## 16. Próximo passo do ciclo (stub) — exportar suspeitas → Linear

**Ainda não implementado neste notebook.** Ideia: comparar a predição do raster (`PREDICT_OUTPUT_PATH`) com os rótulos rasterizados dos geojsons e listar as manchas onde:
- **falso negativo**: o modelo NÃO viu a mancha (recall_por_mancha já ajuda a identificar quais são);
- **falso positivo persistente**: bloco predito como daninha em área SEM polígono anotado — candidato a mancha faltando na anotação.

Essas manchas viram cards em Linear pro anotador revisar. Depois de corrigir, bump `CICLO = 2` no topo e roda tudo de novo — a extração DINOv3 é cacheada (só re-roda pros sites cujos geojsons mudaram), então o próximo ciclo é rápido.

Quando quiser, me peça pra implementar o exportador de suspeitas.